# 🛍️ 範例 03：商場客戶分群 (集群篇) — AI 協作實戰筆記本

本筆記本重現按照 `README.md` 向 AI 提問的成果。初學者可以觀察 AI 如何協助探索無標籤資料、解釋特徵標準化的必要性、運用手肘法與輪廓係數決定最佳群數 K=5，並產出客群商業畫像。

## 階段 0：環境準備與中文字型設定

In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 設定繁體中文字型
try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "sans-serif"]
    plt.rcParams["axes.unicode_minus"] = False

print("環境與中文字型設定完成！")

## 階段 1：資料載入與特徵分佈散佈圖 (EDA)

In [ ]:
df = pd.read_csv("data/customer.csv")
print("=== 資料維度與型態 ===")
df.info()
display(df.head())

# 繪製原始未分群散佈圖
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="收入(千)", y="消費指數(1-100)", s=70, color="royalblue", alpha=0.8)
plt.title("顧客年收入 vs. 消費指數原始分佈散佈圖", fontsize=13)
plt.xlabel("年收入 (千元)", fontsize=11)
plt.ylabel("消費活躍指數 (1-100)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 階段 2：特徵標準化與客觀決定最佳群數 K (手肘法與輪廓係數)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 選取分群特徵並標準化
features = ["收入(千)", "消費指數(1-100)"]
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 測試 K=2 至 K=8
k_range = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# 繪製手肘法與輪廓係數比較圖
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, "bo-", lw=2, markersize=8)
axes[0].set_title("手肘法檢驗 (Inertia 折線圖)", fontsize=13)
axes[0].set_xlabel("分群數量 K")
axes[0].set_ylabel("組內平方和 (Inertia)")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].plot(k_range, silhouette_scores, "rs-", lw=2, markersize=8)
axes[1].set_title("輪廓係數分析 (Silhouette Score)", fontsize=13)
axes[1].set_xlabel("分群數量 K")
axes[1].set_ylabel("輪廓係數")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

print("💡 解讀：當 K=5 時，手肘圖呈現明顯平緩轉折，且輪廓係數達到最高峰 (約 0.555)，故最佳群數為 5！")

## 階段 3：執行 K-Means (K=5) 與分群視覺化 (含質心標記)

In [ ]:
# 訓練 K=5 模型
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df["群別"] = kmeans.fit_predict(X_scaled)

# 將質心還原至原始尺度
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)

# 視覺化 5 群分佈
plt.figure(figsize=(9, 7))
palette = ["#3498db", "#e74c3c", "#2ecc71", "#9b59b6", "#f39c12"]

for cluster_id in range(5):
    subset = df[df["群別"] == cluster_id]
    plt.scatter(subset["收入(千)"], subset["消費指數(1-100)"],
                s=70, color=palette[cluster_id], label=f"群別 {cluster_id}", alpha=0.8)

# 標示群中心
plt.scatter(centroids_original[:, 0], centroids_original[:, 1],
            s=250, c="black", marker="X", edgecolor="yellow", linewidth=2, label="群質心 (Centroid)")

plt.title("K-Means (K=5) 顧客分群視覺化圖表", fontsize=14)
plt.xlabel("年收入 (千元)", fontsize=12)
plt.ylabel("消費指數 (1-100)", fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 階段 4：群集統計分析與商業客群畫像命名

In [ ]:
summary = df.groupby("群別").agg(
    人數=("年齡", "count"),
    平均年齡=("年齡", "mean"),
    平均收入=("收入(千)", "mean"),
    平均消費指數=("消費指數(1-100)", "mean")
).round(1)

persona_names = [
    "平穩中產主力族",
    "年輕活躍衝動族",
    "黃金頂級 VIP 族",
    "節儉保守銀髮族",
    "高收理性觀望族"
]

summary["商業畫像名稱"] = persona_names
display(summary[["商業畫像名稱", "人數", "平均年齡", "平均收入", "平均消費指數"]])

## 階段 5：新顧客即時歸類與推論應用 (Inference)

In [ ]:
def classify_new_customer(scaler, kmeans_model, income, score):
    """輸入新顧客收入與消費指數，回傳群別與專屬行銷建議"""
    new_pt = scaler.transform([[income, score]])
    cluster_id = kmeans_model.predict(new_pt)[0]
    name = persona_names[cluster_id]
    
    strategies = {
        "平穩中產主力族": "推廣定期促銷、滿千送百折價券，維繫日常回購率。",
        "年輕活躍衝動族": "透過社群媒體推播潮牌快閃、聯名限量款，搭配分期無息優惠。",
        "黃金頂級 VIP 族": "指派專屬 VIP 顧問、提供私人鑑賞會與尊榮生日禮遇。",
        "節儉保守銀髮族": "提供民生必需品特惠、長者積點加倍活動。",
        "高收理性觀望族": "強調產品高品質、耐用度與尊爵品牌故事，以高價值感打動理性消費者。"
    }
    return cluster_id, name, strategies[name]

# 測試新會員
test_cases = [
    {"name": "會員甲 (年收入100千, 消費85分)", "income": 100, "score": 85},
    {"name": "會員乙 (年收入25千, 消費15分)", "income": 25, "score": 15}
]

for c in test_cases:
    cid, pname, strat = classify_new_customer(scaler, kmeans, c["income"], c["score"])
    print(f"📌 {c['name']}:")
    print(f"   歸類群別: 群別 {cid} - 【{pname}】")
    print(f"   行銷策略: {strat}\n")